# 07 — Frozen regression evaluation, slices, and bounded errors

**Estimated time:** 60 minutes<br>
**Prerequisites:** 06 — LoRA fine-tuning; prompts and settings are locked<br>
**Learner-produced evidence:** comparable frozen reports, slice tables, and bounded error evidence

## Learning objectives

- Score every method through the same framework-neutral evaluator.
- Keep classification, structured-output, response-policy, and performance evidence separate.
- Use slices and bounded errors without tuning against the frozen set.

This notebook is a teaching interface over the reusable code in `src/`.
It uses only prepared local files. Run `make prepare-flight` before the trip;
no cell installs packages or downloads data.


## Why this matters

This is the first notebook allowed to open the frozen test set. Its job is not to rescue the adapter; it is to estimate how the already-locked methods behave on independent examples. Comparable evaluation requires the same records, contracts, inference settings, and evaluator, plus enough slice and error evidence to explain what a global score hides.

## Key terms in plain language

- **frozen evaluation:** a measurement run whose data, methods, parser, scorers, and thresholds were locked before results.
- **evaluation fingerprint:** a digest of the exact records and evaluation contract used for a result.
- **slice:** a meaningful subset such as an intent, length band, source, or risk category.
- **error taxonomy:** a versioned set of failure categories used to summarize errors consistently.
- **latency:** elapsed time for a defined inference boundary; hardware, warm-up, and batch context affect it.
- **p95 latency:** a value at or above approximately 95 percent of measured latencies, highlighting the slow tail.
- **peak RSS:** the process resident-memory high-water mark; useful context, not a universal GPU or system-memory measure.
- **comparability:** the condition that differences in results can reasonably be attributed to the named method change.


## Mental model — how to think about this

Treat frozen evaluation as the final exam after the student and grading rubric are locked. The test result may support adoption, rejection, or an inconclusive decision. It may also inspire the *next* experiment, but changing the current prompt, adapter, policy, or threshold in response means the old test has become development feedback and a fresh exam is required.

### Running example

A frozen password-related example is now opened for the first formal comparison. Every locked method receives the identical input and is scored by the same parser, label set, response policy, and timing definition. An error can be both `wrong intent` and `invalid schema`; those categories overlap and should not be summed as records.

### Questions to ask before continuing

- Are every method and evaluator consuming the identical ordered example IDs?
- Which metric maps to each real failure cost, and which risks need hard gates?
- Do slice counts support the interpretation, or is an apparently perfect slice tiny?
- Were timing, token, and memory boundaries measured consistently on documented hardware?


## Current best practices

**Guidance reviewed:** 2026-08-01. These are reasons to inspect future tool changes, not a claim that practice stops evolving.

- **Lock methods and contracts before opening test.** Freeze adapter path, prompts, response policy, parsers, label vocabulary, decoding, metrics, and thresholds.
- **Report a metric portfolio.** Include classification metrics, per-label results and counts, schema validity, policy compliance, latency, output tokens, and carefully scoped memory measurements.
- **Pair aggregates with bounded errors.** Use a documented taxonomy and masked samples so reviewers can understand failures without turning evaluation into an uncontrolled data dump.
- **Declare the evaluation scope and verify fingerprints before comparison.** A stratified course-scale subsample and the complete frozen run are different evidence; every compared report must carry the same declared scope and record fingerprint, and only the complete scope is promotion-grade.
- **Preserve uncertainty.** Small slices, missing artifacts, incomparable runs, or unstable execution should lead to an inconclusive decision rather than an overstated win.

## Common mistakes and why they fail

- **Fixing the change after seeing test errors and rerunning the same test.** This optimizes to the exam.
- **Letting one average hide failures.** Macro/weighted metrics and slice tables answer different questions.
- **Interpreting perfect performance on a tiny slice as certainty.** Always display the denominator and limitations.
- **Quoting resource numbers without scope.** Peak RSS is not GPU peak memory, and timings from different machines or warm-up states are not directly comparable.

### What kind of guidance is this?

A **specification** defines a technical contract; **tool guidance** describes current official library behavior; **risk guidance** is voluntary governance guidance; and a **course rule** is this project's deliberately conservative choice. Do not call all four a formal standard. The lesson is complete offline; these primary links are optional follow-up reading.

- **Tool guidance:** [MLflow evaluation datasets and regression-test guidance](https://mlflow.org/docs/latest/genai/datasets/)
- **Tool guidance:** [scikit-learn metrics and scoring guide](https://scikit-learn.org/stable/modules/model_evaluation.html)


## Setup — run, do not edit

Run the next cell once. It verifies the dedicated local Python kernel, finds
this sample project, and enables supported offline flags **before** model or
tracking libraries are imported. A successful cell ends with `setup: ready`.

This is one defense layer, not proof that every native library is physically
incapable of networking. The flight-preparation manifest, cached assets,
socket-denial checks, and a Wi-Fi-off rehearsal provide the other layers.


In [ ]:
import sys
from importlib import import_module
from pathlib import Path

current = Path.cwd().resolve()
project_root = None
for candidate in (current, *current.parents):
    direct = candidate
    nested = candidate / "examples" / "local-finetuning"
    if (direct / "src" / "aai_local_finetuning").is_dir():
        project_root = direct
        break
    if (nested / "src" / "aai_local_finetuning").is_dir():
        project_root = nested
        break
if project_root is None:
    raise RuntimeError(
        "Cannot locate examples/local-finetuning. Open this notebook from the "
        "repository, or run `make notebook` from the repository root."
    )

expected_python = (project_root / ".venv" / "bin" / "python").resolve()
active_python = Path(sys.executable).resolve()
if not expected_python.is_file() or active_python != expected_python:
    raise RuntimeError(
        "Wrong notebook kernel. Run `make notebook` from the repository root, "
        "then select 'AAI Local Fine-Tuning (offline)'. "
        f"Active Python: {active_python}; expected: {expected_python}"
    )

source_root = str(project_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

enable_offline_environment = import_module(
    "aai_local_finetuning.offline"
).enable_offline_environment
enable_offline_environment()

{
    "setup": "ready",
    "kernel": "AAI Local Fine-Tuning (offline)",
    "python": str(active_python),
    "network_library_flags": "enabled",
    "note": "Continue to the lesson; this cell is setup, not an exercise.",
}

## Lock the decision contract before opening test

These versioned **course defaults** require at least a 0.01 absolute
macro-F1 gain over the strongest meaningful baseline, schema validity
≥ 0.98, response-policy compliance ≥ 0.95, and zero unsupported labels.
They are fixed now, before any test row or result is loaded.

Category accuracy, escalation accuracy, latency, input/output tokens,
peak memory, and adapter size are still reported but are **observational,
not gating**, because this learning project has not invented business
budgets for them. A production owner must set risk-based non-regression
and resource gates. This compact course also uses point estimates; a
higher-stakes decision should add paired uncertainty analysis and, for
training variance, repeated seeds.


In [ ]:
from aai_local_finetuning.evaluation import PromotionThresholds

locked_thresholds = PromotionThresholds()
locked_thresholds.model_dump(mode="json")

## Open the frozen boundary once choices are locked

This is the first notebook that reads test examples for scoring. Do not
revise prompts, demonstrations, thresholds, response policy, or training
configuration after seeing these errors.

The default evaluation scope is a **deterministic stratified
subsample**: the first two frozen records of every supported intent, in
frozen-split order (54 of the 270 test records). Every intent keeps
support, so macro-F1 is defined for each of the 27 classes and the
six-method comparison is honest at course scale — unlike a first-N
slice, which would leave most intents with zero support and make the
macro average meaningless. Notebook 08 reads this same scope by default
and reaches a real adopt/reject decision at course scale.
**Promotion-grade evidence still requires the complete run**: set
`EVALUATION_SUBSAMPLE_PER_INTENT = None` when you have the time and
battery for all 270 records.


In [ ]:
import pandas as pd

from aai_local_finetuning.evaluation import (
    DeterministicInferenceConfig,
    EvaluationReport,
    KeywordRuleBaseline,
    MajorityBaseline,
    build_local_mlx_inference_config,
    evaluate_predictions,
    format_error_analysis,
    recheck_evaluation_session,
    start_evaluation_session,
    write_predictions_jsonl,
    write_report_json,
)
from aai_local_finetuning.learning import (
    COMPLETE_EVALUATION_SCOPE,
    generate_support_predictions,
    load_support_splits,
    report_row,
    stratified_evaluation_scope,
    stratified_subsample,
    support_contract,
)
from aai_local_finetuning.modeling import LocalMLXPredictor
from aai_local_finetuning.offline import verify_flight_manifest
from aai_local_finetuning.settings import PROJECT_ROOT, load_settings
from aai_local_finetuning.training import (
    TrainingManifestError,
    recheck_training_snapshot,
    require_valid_training_snapshot,
    shared_adapter_lock,
)

EVALUATION_SUBSAMPLE_PER_INTENT = 2  # Set to None for complete promotion evidence.
scope = (
    COMPLETE_EVALUATION_SCOPE
    if EVALUATION_SUBSAMPLE_PER_INTENT is None
    else stratified_evaluation_scope(EVALUATION_SUBSAMPLE_PER_INTENT)
)
evidence_dir = PROJECT_ROOT / "artifacts" / "notebook" / "evaluation"
evidence_dir.mkdir(parents=True, exist_ok=True)
lineage_copy = evidence_dir / f"{scope}-lora-change-training-manifest.json"
lora_prediction_path = evidence_dir / f"{scope}-lora-change-predictions.jsonl"
lora_report_path = evidence_dir / f"{scope}-lora-change-report.json"
persisted_method_names = (
    "majority",
    "keyword-rule",
    "basic",
    "strong",
    "few_shot",
)
baseline_evidence_paths = tuple(
    path
    for name in persisted_method_names
    for path in (
        evidence_dir / f"{scope}-{name}-predictions.jsonl",
        evidence_dir / f"{scope}-{name}-report.json",
    )
)
for stale_path in (
    *baseline_evidence_paths,
    lineage_copy,
    lora_prediction_path,
    lora_report_path,
):
    stale_path.unlink(missing_ok=True)

settings = load_settings()
verify_flight_manifest(settings)
splits = load_support_splits(settings)
allowed_intents, _ = support_contract(splits.train)
FULL_FROZEN_COUNT = len(splits.test)
frozen_records = (
    splits.test
    if EVALUATION_SUBSAMPLE_PER_INTENT is None
    else stratified_subsample(
        splits.test,
        per_intent=EVALUATION_SUBSAMPLE_PER_INTENT,
    )
)
{
    "evaluation_scope": scope,
    "scored_now": len(frozen_records),
    "full_frozen_count": FULL_FROZEN_COUNT,
    "intents_with_support": len({record.target.intent for record in frozen_records}),
    "supported_intents": len(allowed_intents),
    "promotion_grade": scope == COMPLETE_EVALUATION_SCOPE,
}

## Recompute deterministic methods on the same records

Comparisons require the same record IDs, supported labels, and evaluator.
The majority method is a sanity floor; the keyword/rule method is a
meaningful transparent baseline.


In [ ]:
frozen_evaluation_session = start_evaluation_session(settings)
methods = {
    "majority": MajorityBaseline.fit(splits.train).predict_many(frozen_records),
    "keyword-rule": KeywordRuleBaseline.fit(splits.train).predict_many(frozen_records),
}
reports = {
    name: evaluate_predictions(
        frozen_records,
        predictions,
        supported_intents=allowed_intents,
        evaluation_session=frozen_evaluation_session,
        inference_config=DeterministicInferenceConfig(method=name),
    )
    for name, predictions in methods.items()
}
recheck_evaluation_session(frozen_evaluation_session)
pd.DataFrame([report_row(name, report) for name, report in reports.items()])

## Evaluate the three untouched-model prompts

One local predictor keeps model weights fixed. Each method receives the
same records and maximum output budget. The few-shot helper draws only
from train. This can take several seconds on the prepared Mac.


In [ ]:
predictor = LocalMLXPredictor(settings.model_dir)
for strategy in ("basic", "strong", "few_shot"):
    inference_config = build_local_mlx_inference_config(
        frozen_evaluation_session,
        method=strategy,
        prompt_recipe=strategy,
        max_tokens=96,
        few_shot_examples=(4 if strategy == "few_shot" else 0),
    )
    predictions = generate_support_predictions(
        predictor,
        frozen_records,
        strategy=strategy,
        train_records=splits.train,
        inference_config=inference_config,
    )
    methods[strategy] = predictions
    reports[strategy] = evaluate_predictions(
        frozen_records,
        predictions,
        supported_intents=allowed_intents,
        evaluation_session=frozen_evaluation_session,
        inference_config=inference_config,
    )
recheck_evaluation_session(frozen_evaluation_session)
pd.DataFrame([report_row(name, report) for name, report in reports.items()])

## Add the LoRA change and persist its evidence atomically

Notebook probes never overwrite official evaluation artifacts. Filenames
carry the declared evaluation scope — `stratified-subsample-2-per-intent`
by default, `complete` for the full frozen run — and notebook 08 reads
the same scope by default, so the evidence this cell writes is exactly
what the decision notebook consumes. Reports carry the evaluation
fingerprint used to prove comparability later.

The canonical adapter is separate from notebook smoke adapters. A weight
file alone is not training lineage: the success manifest must also match
the current adapter bytes, adapter configuration, full training YAML,
effective settings, expected base-model revision and files, and every
prepared training-data file. It also binds the training-time source,
interpreter/platform, and exact package set. Every baseline and change
report separately records the evaluation-time execution-contract hash.
Its inference configuration records the exact base-model file inventory,
prompt recipe, greedy decoder defaults, output budget, and adapter
manifest. Unverified additions that a loader could consume—such as another
weight shard or chat template—are rejected. The evaluation session was
captured before model construction and remains open through scoring and
evidence persistence; its final recheck detects even a source, package, or
model-file change that was restored before the write completed. Training
lineage is added by reconstructing the strict report, so its paired-lineage
validator runs again before anything is written.
One shared adapter lock covers validation, prediction, scoring, report
writes, and the exact manifest copy. Training cannot replace the adapter
during any part of that evidence chain. Missing or stale evidence keeps
the later decision inconclusive.

Same-name notebook artifacts were invalidated before the first method was
fitted. If setup, prediction, or scoring fails, an older file from this
scope therefore cannot masquerade as evidence from this attempt.


In [ ]:
try:
    for name, report in reports.items():
        write_predictions_jsonl(
            evidence_dir / f"{scope}-{name}-predictions.jsonl",
            methods[name],
        )
        write_report_json(
            evidence_dir / f"{scope}-{name}-report.json",
            report,
        )
    recheck_evaluation_session(frozen_evaluation_session)
except (OSError, RuntimeError, ValueError):
    for incomplete_path in baseline_evidence_paths:
        incomplete_path.unlink(missing_ok=True)
    raise

adapter_weights = settings.adapter_dir / "adapters.safetensors"
adapter_snapshot = None
if adapter_weights.is_file():
    try:
        with shared_adapter_lock(settings.adapter_dir):
            adapter_snapshot = require_valid_training_snapshot(
                settings.adapter_dir,
                config_path=(PROJECT_ROOT / "configs" / "training" / "lora.yaml"),
            )
            lora_evaluation_session = start_evaluation_session(settings)
            lora_inference_config = build_local_mlx_inference_config(
                lora_evaluation_session,
                method="lora-change",
                prompt_recipe="strong",
                max_tokens=96,
                adapter_manifest_sha256=(adapter_snapshot.manifest_sha256),
            )
            lora_predictor = LocalMLXPredictor(
                settings.model_dir,
                adapter_path=settings.adapter_dir,
            )
            methods["lora-change"] = generate_support_predictions(
                lora_predictor,
                frozen_records,
                strategy="strong",
                train_records=splits.train,
                inference_config=lora_inference_config,
            )
            recheck_training_snapshot(adapter_snapshot)
            unbound_lora_report = evaluate_predictions(
                frozen_records,
                methods["lora-change"],
                supported_intents=allowed_intents,
                evaluation_session=lora_evaluation_session,
                inference_config=lora_inference_config,
            )
            reports["lora-change"] = EvaluationReport.model_validate(
                unbound_lora_report.model_dump(mode="python")
                | {
                    "training_manifest_sha256": (adapter_snapshot.manifest_sha256),
                    "training_execution_contract_sha256": (
                        adapter_snapshot.manifest.execution_contract_sha256
                    ),
                },
            )
            write_predictions_jsonl(
                lora_prediction_path,
                methods["lora-change"],
            )
            write_report_json(
                lora_report_path,
                reports["lora-change"],
            )
            lineage_copy.write_bytes(adapter_snapshot.raw_manifest_bytes)
            recheck_training_snapshot(adapter_snapshot)
            recheck_evaluation_session(lora_evaluation_session)
    except (
        OSError,
        RuntimeError,
        ValueError,
        TrainingManifestError,
    ) as error:
        adapter_snapshot = None
        methods.pop("lora-change", None)
        reports.pop("lora-change", None)
        for incomplete_path in (
            lineage_copy,
            lora_prediction_path,
            lora_report_path,
        ):
            incomplete_path.unlink(missing_ok=True)
        print(f"Canonical LoRA adapter ignored: {error}")
else:
    print("Canonical LoRA adapter absent; change evidence is incomplete.")

display(pd.DataFrame([report_row(name, report) for name, report in reports.items()]))
{
    "evaluation_execution_contract_sha256": sorted(
        {report.evaluation_execution_contract_sha256 for report in reports.values()}
    ),
    "artifacts": sorted(path.name for path in evidence_dir.glob(f"{scope}-*")),
}

## Slice and error analysis

Counts accompany every slice so a perfect score on one example is not
overinterpreted. Peak RSS is a process high-water mark, not precise
per-request memory. Error previews are bounded and already masked.


In [ ]:
inspected_method = "lora-change" if "lora-change" in reports else "strong"
inspected_report = reports[inspected_method]
lowest_intents = (
    pd.DataFrame(
        [
            {"intent": intent, "f1": score}
            for intent, score in inspected_report.classification.per_intent_f1.items()
        ]
    )
    .sort_values("f1")
    .head(10)
)
difficulty_slices = pd.DataFrame(
    [
        {"difficulty": name, **metrics.model_dump(mode="json")}
        for name, metrics in inspected_report.by_difficulty.items()
    ]
)
lowest_intents, difficulty_slices

In [ ]:
print(format_error_analysis(inspected_report))

## Exercise — write a non-tuning error conclusion

Select one error kind and explain what it means. Success means you do not
propose changing the prompt or model based on frozen evidence; propose a
future experiment with a newly versioned evaluation boundary instead.


In [ ]:
frozen_error_conclusion = (
    "Record schema failures as a result of this locked experiment. Any "
    "remediation becomes a new change evaluated on a new untouched test version."
)
assert "new" in frozen_error_conclusion.lower()
frozen_error_conclusion

**Hint:** frozen errors are evidence about this experiment, not free
development feedback for the same test version.


## Checkpoint

Confirm that every compared report has the same fingerprint and record
count. The default stratified scope supports a real course-scale
decision in notebook 08; only the `complete` scope is promotion-grade
evidence for a real deployment decision.

**Next:** `08_mlflow_and_promotion.ipynb` records lineage and computes an
adopt, reject, or inconclusive decision.
